# Diabetes Risk Classification Using Machine Learning Classifiers
**Course**: Machine Learning (AIMLZG565) - Assignment 2  
**Student Name**: Mayank Gupta  
**BITS ID**: 2026AK05256  
**Random State**: 26  

---

This notebook trains and evaluates the 5 required classification models:
1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **K-Nearest Neighbors (KNN)**
4. **Gaussian Naive Bayes**
5. **Support Vector Machine (SVC RBF)**

Across the 6 mandatory evaluation metrics: Accuracy, ROC-AUC, Precision, Recall, F1 Score, and MCC.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

RANDOM_STATE = 26
DATASET_PATH = 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv'
MODEL_DIR = 'model'
TEST_DATA_PATH = 'test_data.csv'

os.makedirs(MODEL_DIR, exist_ok=True)
print('Libraries imported successfully.')

## 1. Data Ingestion & Validation

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')

target_col = 'Diabetes_binary'
feature_cols = [col for col in df.columns if col != target_col]
print(f'Total predictor features ({len(feature_cols)}):', feature_cols)

print('\nTarget Class Distribution:')
print(df[target_col].value_counts())

X = df[feature_cols]
y = df[target_col].astype(int)

## 2. Train-Test Split (80/20 Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

# Export curated balanced test sample (1,500 rows) for Streamlit upload
test_df_full = X_test.copy()
test_df_full[target_col] = y_test
test_0 = test_df_full[test_df_full[target_col] == 0].sample(n=750, random_state=RANDOM_STATE)
test_1 = test_df_full[test_df_full[target_col] == 1].sample(n=750, random_state=RANDOM_STATE)
test_sample = pd.concat([test_0, test_1]).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
test_sample.to_csv(TEST_DATA_PATH, index=False)
print(f'Exported {TEST_DATA_PATH} with shape: {test_sample.shape}')

## 3. Preprocessing Pipeline (StandardScaler)

In [ ]:
scale_features = ['BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Education', 'Income']
binary_features = [col for col in feature_cols if col not in scale_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), scale_features),
        ('pass', 'passthrough', binary_features)
    ],
    remainder='drop'
)

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

joblib.dump(preprocessor, os.path.join(MODEL_DIR, 'preprocessor.joblib'))
print('Preprocessor fitted on training data and saved to model/preprocessor.joblib.')

## 4. Train 5 Classifiers & Calculate 6 Evaluation Metrics

In [ ]:
svm_indices = X_train.sample(n=15000, random_state=RANDOM_STATE).index
X_train_svm = preprocessor.transform(X_train.loc[svm_indices])
y_train_svm = y_train.loc[svm_indices]

models = {
    'Logistic Regression': {
        'estimator': LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
        'X_train': X_train_trans,
        'y_train': y_train,
        'filename': 'logistic_regression.joblib'
    },
    'Decision Tree': {
        'estimator': DecisionTreeClassifier(max_depth=6, min_samples_split=20, min_samples_leaf=10, random_state=RANDOM_STATE),
        'X_train': X_train_trans,
        'y_train': y_train,
        'filename': 'decision_tree.joblib'
    },
    'KNN': {
        'estimator': KNeighborsClassifier(n_neighbors=9, metric='euclidean', n_jobs=-1),
        'X_train': X_train_trans,
        'y_train': y_train,
        'filename': 'knn.joblib'
    },
    'Naive Bayes': {
        'estimator': GaussianNB(),
        'X_train': X_train_trans,
        'y_train': y_train,
        'filename': 'naive_bayes.joblib'
    },
    'Support Vector Machine': {
        'estimator': SVC(kernel='rbf', C=1.0, probability=True, random_state=RANDOM_STATE, cache_size=1000),
        'X_train': X_train_svm,
        'y_train': y_train_svm,
        'filename': 'svm.joblib'
    }
}

results = []
for name, config in models.items():
    print(f'Training {name}...')
    mdl = config['estimator']
    mdl.fit(config['X_train'], config['y_train'])
    joblib.dump(mdl, os.path.join(MODEL_DIR, config['filename']))
    
    y_pred = mdl.predict(X_test_trans)
    y_proba = mdl.predict_proba(X_test_trans)[:, 1] if hasattr(mdl, 'predict_proba') else None
    
    results.append({
        'ML Model Name': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_proba), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4)
    })

results_df = pd.DataFrame(results)
print('\n' + '=' * 80)
print('EVALUATION COMPARISON TABLE (Mandatory Step 5d)')
print('=' * 80)
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(MODEL_DIR, 'evaluation_metrics.csv'), index=False)